# Cross-Dataset Evaluation: Train on UTD-MHAD → Test on CZU-MHAD

**Experimental setup:**
- **Train/Val**: UTD-MHAD dataset — all subjects except 1 (validation), using **depth + sensor + skeleton** modalities only (no video)
- **Test**: CZU-MHAD dataset — entire dataset used as held-out test set
- **Feature selection**: learned on UTD-MHAD (train/val), then applied to both UTD and CZU feature spaces
- **Comparison**: Baseline (no feature selection) vs. 14 EvoloPy metaheuristics


In [1]:
import torch
print(torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

1
NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
from pathlib import Path
from copy import deepcopy
import random
import sys
import gc
import json as json_lib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_score, recall_score, confusion_matrix)
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

sys.path.append('../EvoloPy-master')
from EvoloPy.optimizers import BAT, CS, DE, FFA, GA, GWO, HHO, JAYA, MFO, MVO, PSO, SCA, SSA, WOA

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## Configuration

In [3]:
UTD_FEATURE_DIR = Path("features")
CZU_FEATURE_DIR = Path("../czu-mhad/features")

RESULTS_ROOT = Path("results_cross_dataset_czu-mhad")
RESULTS_ROOT.mkdir(exist_ok=True)
PLOTS_DIR = RESULTS_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

SHARED_MODALITY_KEYS  = ['depth_feat', 'sensor_feat', 'skeleton_feat']
SHARED_MODALITY_NAMES = ['depth', 'sensor', 'skeleton']

N_EPOCHS       = 300
BATCH_SIZE     = 32
LEARNING_RATE  = 0.001
DEPTH_PCA_DIM  = 512
DEPTH_DR_METHOD = 'kernel_pca'

N_POPULATION   = 20
MAX_ITERATIONS = 30

TARGET_FEATURE_PERCENTAGE = 0.5

EVOLOPY_OPTIMIZERS = {
    'BAT':  BAT.BAT,
    'CS':   CS.CS,
    'DE':   DE.DE,
    'FFA':  FFA.FFA,
    'GA':   GA.GA,
    'GWO':  GWO.GWO,
    'HHO':  HHO.HHO,
    'JAYA': JAYA.JAYA,
    'MFO':  MFO.MFO,
    'MVO':  MVO.MVO,
    'PSO':  PSO.PSO,
    'SCA':  SCA.SCA,
    'SSA':  SSA.SSA,
    'WOA':  WOA.WOA,
}

META_METHODS = ['baseline'] + [f'meta_{k}' for k in EVOLOPY_OPTIMIZERS.keys()]

for method in META_METHODS:
    (RESULTS_ROOT / method).mkdir(exist_ok=True)

print(f"Shared modalities: {SHARED_MODALITY_NAMES}")
print(f"Methods: {META_METHODS}")
print(f"N_EPOCHS: {N_EPOCHS}, N_POPULATION: {N_POPULATION}, MAX_ITERATIONS: {MAX_ITERATIONS}")

Shared modalities: ['depth', 'sensor', 'skeleton']
Methods: ['baseline', 'meta_BAT', 'meta_CS', 'meta_DE', 'meta_FFA', 'meta_GA', 'meta_GWO', 'meta_HHO', 'meta_JAYA', 'meta_MFO', 'meta_MVO', 'meta_PSO', 'meta_SCA', 'meta_SSA', 'meta_WOA']
N_EPOCHS: 300, N_POPULATION: 20, MAX_ITERATIONS: 30


## Load Data

In [4]:
X_feat_utd  = joblib.load(UTD_FEATURE_DIR / "X_feat.pkl")
y_utd       = np.load(UTD_FEATURE_DIR / "y.npy")
subjects_utd = np.load(UTD_FEATURE_DIR / "subjects.npy")
le_utd      = joblib.load(UTD_FEATURE_DIR / "label_encoder.pkl")

print(f"UTD-MHAD: {len(X_feat_utd)} samples, "
      f"{len(np.unique(y_utd))} classes, "
      f"{len(np.unique(subjects_utd))} subjects")

X_utd = {}
for key, name in zip(SHARED_MODALITY_KEYS, SHARED_MODALITY_NAMES):
    X_utd[name] = np.array([s[key] for s in X_feat_utd])
    print(f"  UTD {name}: {X_utd[name].shape}")

UTD-MHAD: 861 samples, 27 classes, 8 subjects
  UTD depth: (861, 5508)
  UTD sensor: (861, 652)
  UTD skeleton: (861, 1879)


In [6]:
X_feat_czu  = joblib.load(CZU_FEATURE_DIR / "X_feat.pkl")
y_czu       = np.load(CZU_FEATURE_DIR / "y.npy")
le_czu      = joblib.load(CZU_FEATURE_DIR / "label_encoder.pkl")

print(f"CZU-MHAD: {len(X_feat_czu)} samples, "
      f"{len(np.unique(y_czu))} classes")

X_czu = {}
for key, name in zip(SHARED_MODALITY_KEYS, SHARED_MODALITY_NAMES):
    X_czu[name] = np.array([s[key] for s in X_feat_czu])
    print(f"  CZU {name}: {X_czu[name].shape}")

for name in SHARED_MODALITY_NAMES:
    utd_dim = X_utd[name].shape[1]
    czu_dim = X_czu[name].shape[1]
    match = "✓" if utd_dim == czu_dim else "✗ MISMATCH"
    print(f"  {name}: UTD={utd_dim}, CZU={czu_dim} {match}")

CZU-MHAD: 1165 samples, 22 classes
  CZU depth: (1165, 5508)
  CZU sensor: (1165, 652)
  CZU skeleton: (1165, 1879)
  depth: UTD=5508, CZU=5508 ✓
  sensor: UTD=652, CZU=652 ✓
  skeleton: UTD=1879, CZU=1879 ✓


In [12]:
utd_classes = list(le_utd.classes_)
print(f"UTD-MHAD classes ({len(utd_classes)}): {utd_classes}")

utd_clap_idx     = utd_classes.index(4)
utd_draw_cw_idx  = utd_classes.index(9)
utd_draw_ccw_idx = utd_classes.index(10)

print(f"\nUTD matching indices:")
print(f"  clap:            {utd_clap_idx}  ('{utd_classes[utd_clap_idx]}')")
print(f"  draw circle CW:  {utd_draw_cw_idx}  ('{utd_classes[utd_draw_cw_idx]}')")
print(f"  draw circle CCW: {utd_draw_ccw_idx}  ('{utd_classes[utd_draw_ccw_idx]}')")

UTD_NUM_CLASSES = len(utd_classes)

# CZU-MHAD label encoder
czu_classes = list(le_czu.classes_)
print(f"\nCZU-MHAD classes ({len(czu_classes)}): {czu_classes}")

czu_clap_idx         = 14
czu_circle_right_idx = 8
czu_circle_left_idx  = 9

print(f"\nCZU matching indices:")
print(f"  clap:         {czu_clap_idx}  ('{czu_classes[czu_clap_idx]}')")
print(f"  circle right: {czu_circle_right_idx}  ('{czu_classes[czu_circle_right_idx]}')")
print(f"  circle left:  {czu_circle_left_idx}  ('{czu_classes[czu_circle_left_idx]}')")


# Merged 2-class label space
LABEL_CLAP         = 0
LABEL_DRAW_CIRCLE  = 1
MERGED_CLASS_NAMES = ['clap', 'draw_circle']
NUM_CLASSES        = 2

# UTD model output index → merged label (-1 = outside, discarded at eval time)
utd_to_merged = {
    utd_clap_idx:     LABEL_CLAP,
    utd_draw_cw_idx:  LABEL_DRAW_CIRCLE,
    utd_draw_ccw_idx: LABEL_DRAW_CIRCLE,
}

# CZU encoded label → merged label
czu_to_merged = {
    czu_clap_idx:         LABEL_CLAP,
    czu_circle_right_idx: LABEL_DRAW_CIRCLE,
    czu_circle_left_idx:  LABEL_DRAW_CIRCLE,
}

print(f"\nMerged 2-class mapping:")
print(f"  UTD  clap ({utd_clap_idx})           → {LABEL_CLAP}  (clap)")
print(f"  UTD  draw CW ({utd_draw_cw_idx})         → {LABEL_DRAW_CIRCLE}  (draw_circle)")
print(f"  UTD  draw CCW ({utd_draw_ccw_idx})        → {LABEL_DRAW_CIRCLE}  (draw_circle)")
print(f"  CZU  clap ({czu_clap_idx})          → {LABEL_CLAP}  (clap)")
print(f"  CZU  circle right ({czu_circle_right_idx})  → {LABEL_DRAW_CIRCLE}  (draw_circle)")
print(f"  CZU  circle left ({czu_circle_left_idx})   → {LABEL_DRAW_CIRCLE}  (draw_circle)")
print(f"  UTD  all other classes         → -1  (discarded)")

# Filter CZU to only the 2 matching classes
keep_mask = np.isin(y_czu, list(czu_to_merged.keys()))
X_czu = {name: X_czu[name][keep_mask] for name in SHARED_MODALITY_NAMES}
y_czu_remapped = np.array([czu_to_merged[yi] for yi in y_czu[keep_mask]])

print(f"\nCZU samples kept: {keep_mask.sum()} / {len(keep_mask)}")
print(f"  clap:        {(y_czu_remapped == LABEL_CLAP).sum()}")
print(f"  draw_circle: {(y_czu_remapped == LABEL_DRAW_CIRCLE).sum()}")

UTD-MHAD classes (27): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27)]

UTD matching indices:
  clap:            3  ('4')
  draw circle CW:  8  ('9')
  draw circle CCW: 9  ('10')

CZU-MHAD classes (22): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22)]

CZU matching indices:
  clap:         14  ('15')
  circle right: 8  ('9')
  circle left:  9  ('10')

Merged 2-class mapping:
  UTD  clap (3)           → 0  (clap)
  UTD  draw

## Model & Helper Functions

In [7]:
class MultiModalDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels   = torch.LongTensor(labels)
    def __len__(self):  return len(self.labels)
    def __getitem__(self, idx): return self.features[idx], self.labels[idx]


class SimpleNN(nn.Module):
    """MLP — adaptive hidden sizes based on input dim"""
    def __init__(self, input_dim, num_classes):
        super().__init__()
        hidden1 = max(128, min(512, input_dim * 2))
        hidden2 = max(64,  min(256, hidden1 // 2))
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden2, num_classes)
        )
    def forward(self, x): return self.classifier(x)


def count_model_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model_size_mb(model):
    p = sum(p.nelement() * p.element_size() for p in model.parameters())
    b = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (p + b) / (1024 ** 2)

def get_gpu_memory_mb():
    return torch.cuda.memory_allocated() / (1024**2) if torch.cuda.is_available() else 0.0

def get_dataset_size_mb(X):
    return X.nbytes / (1024**2)

print("Model and helpers defined.")

Model and helpers defined.


In [8]:
def prepare_data(X_per_modality, train_idx, val_idx, depth_pca_dim=None):
    """
    Normalize per modality (fit on train), optionally apply KernelPCA on depth.
    Returns X_train, X_val, feature_dims, scalers, pca_obj
    Scalers and pca_obj are returned so they can be applied to CZU test data.
    """
    modality_train, modality_val = {}, {}
    scalers  = {}
    pca_obj  = None
    feature_dims = {}

    for name in SHARED_MODALITY_NAMES:
        X_mod = X_per_modality[name]
        X_tr  = X_mod[train_idx]
        X_v   = X_mod[val_idx]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_v  = scaler.transform(X_v)
        scalers[name] = scaler

        if name == 'depth' and depth_pca_dim and depth_pca_dim < X_tr.shape[1]:
            pca_obj = KernelPCA(n_components=depth_pca_dim, kernel='rbf',
                                random_state=42, n_jobs=-1)
            X_tr = pca_obj.fit_transform(X_tr)
            X_v  = pca_obj.transform(X_v)
            print(f"    KernelPCA depth: {X_per_modality[name].shape[1]} -> {depth_pca_dim}")

        modality_train[name] = X_tr
        modality_val[name]   = X_v
        feature_dims[name]   = X_tr.shape[1]

    X_train = np.concatenate([modality_train[n] for n in SHARED_MODALITY_NAMES], axis=1)
    X_val   = np.concatenate([modality_val[n]   for n in SHARED_MODALITY_NAMES], axis=1)

    total = sum(feature_dims.values())
    print(f"    Dims: " + " | ".join(f"{n}={feature_dims[n]}" for n in SHARED_MODALITY_NAMES)
          + f" | TOTAL={total}")
    return X_train, X_val, feature_dims, scalers, pca_obj


def adapt_scaler_to_target(utd_scaler, X_czu_raw):
    """
    Adapt a UTD-fitted StandardScaler to the CZU domain.
    Strategy: replace mean_ and scale_ with CZU statistics,
    so CZU features are normalized in their own distribution
    while the same feature space structure is preserved.
    """
    from copy import deepcopy
    adapted = deepcopy(utd_scaler)
    adapted.mean_  = np.mean(X_czu_raw, axis=0)
    adapted.scale_ = np.std(X_czu_raw,  axis=0) + 1e-8  # avoid division by zero
    adapted.var_   = np.var(X_czu_raw,  axis=0)
    return adapted


def apply_preprocessing_to_czu(X_czu_per_modality, utd_scalers, pca_obj):
    """
    Apply domain-adapted preprocessing to CZU test set.
    Scaler mean/scale are re-estimated from CZU data (per modality),
    so CZU features are normalized in their own distribution.
    PCA projection (fitted on UTD) is kept as-is.
    """
    modality_arrays = {}
    for name in SHARED_MODALITY_NAMES:
        X_raw = X_czu_per_modality[name]

        # Adapt the UTD scaler to CZU's own mean/std
        adapted_scaler = adapt_scaler_to_target(utd_scalers[name], X_raw)
        X = adapted_scaler.transform(X_raw)

        # PCA projection stays the same (learned on UTD)
        if name == 'depth' and pca_obj is not None:
            X = pca_obj.transform(X)

        modality_arrays[name] = X

    return np.concatenate([modality_arrays[n] for n in SHARED_MODALITY_NAMES], axis=1)


def calculate_modality_retention(binary_mask, feature_dims):
    start_idx = 0
    retention = {}
    for modality in SHARED_MODALITY_NAMES:
        dim     = feature_dims[modality]
        end_idx = start_idx + dim
        sel     = np.sum(binary_mask[start_idx:end_idx])
        retention[modality] = {
            'selected': int(sel), 'total': dim,
            'percentage': float(sel / dim * 100)
        }
        start_idx = end_idx
    return retention

print("Data preparation functions defined.")

Data preparation functions defined.


In [9]:
def train_model(model, train_loader, val_loader, num_epochs, lr):
    """Train with early-stopping on val acc; return best model state and metrics."""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    gpu_mem_before = get_gpu_memory_mb()
    t0 = time.time()
    train_losses = []
    best_val_acc, best_state = -1.0, None

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        for feats, labels in train_loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(feats), labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        train_losses.append(epoch_loss / len(train_loader))

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for feats, labels in val_loader:
                preds = torch.argmax(model(feats.to(DEVICE)), dim=1)
                correct += (preds.cpu() == labels).sum().item()
                total   += labels.size(0)
        if correct / total > best_val_acc:
            best_val_acc = correct / total
            best_state   = deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, {
        'train_time_sec': time.time() - t0,
        'best_val_acc': best_val_acc,
        'train_losses': train_losses,
        'gpu_mem_peak_mb': torch.cuda.max_memory_allocated() / (1024**2)
                           if torch.cuda.is_available() else 0,
    }


def evaluate_loader(model, loader):
    """Return preds and true labels for any DataLoader."""
    model.eval()
    preds_all, true_all = [], []
    with torch.no_grad():
        for feats, labels in loader:
            p = torch.argmax(model(feats.to(DEVICE)), dim=1).cpu().numpy()
            preds_all.extend(p)
            true_all.extend(labels.numpy())
    return np.array(preds_all), np.array(true_all)


def evaluate_czu_with_logit_mask(model, X_czu_scaled, utd_num_classes):
    """
    Evaluate the UTD-trained model on CZU data using logit masking.

    The model outputs logits over all UTD_NUM_CLASSES (e.g. 27).
    We mask all non-matching UTD classes to -inf so argmax can only
    land on clap / draw_circle_CW / draw_circle_CCW.
    Those raw UTD predictions are then merged into the 2-class space.

    Returns merged_preds (0=clap, 1=draw_circle) aligned with y_czu_remapped.
    """
    # Build logit mask: -inf everywhere except the 3 matching UTD classes
    logit_mask = torch.full((utd_num_classes,), float('-inf'), device=DEVICE)
    for c in [utd_clap_idx, utd_draw_cw_idx, utd_draw_ccw_idx]:
        logit_mask[c] = 0.0

    model.eval()
    raw_preds = []
    with torch.no_grad():
        for i in range(len(X_czu_scaled)):
            sample = torch.FloatTensor(X_czu_scaled[i:i+1]).to(DEVICE)
            output = model(sample)
            masked = output + logit_mask   # kills all non-matching classes
            pred   = torch.argmax(masked, dim=1).cpu().item()
            raw_preds.append(pred)

    raw_preds = np.array(raw_preds)

    # Map UTD class indices → merged 2-class labels
    merged_preds = np.full(len(raw_preds), -1, dtype=int)
    merged_preds[raw_preds == utd_clap_idx]     = LABEL_CLAP
    merged_preds[raw_preds == utd_draw_cw_idx]  = LABEL_DRAW_CIRCLE
    merged_preds[raw_preds == utd_draw_ccw_idx] = LABEL_DRAW_CIRCLE

    return merged_preds, raw_preds


def compute_metrics(preds, true):
    return {
        'accuracy':         float(accuracy_score(true, preds)),
        'f1_macro':         float(f1_score(true, preds, average='macro',    zero_division=0)),
        'f1_weighted':      float(f1_score(true, preds, average='weighted', zero_division=0)),
        'precision_macro':  float(precision_score(true, preds, average='macro',    zero_division=0)),
        'recall_macro':     float(recall_score(true, preds, average='macro',    zero_division=0)),
    }

print("Training and evaluation functions defined.")

Training and evaluation functions defined.


In [10]:
def train_model_quick(model, train_loader, val_loader, epochs, lr, device):
    """Lightweight training for metaheuristic fitness evaluation."""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_val_loss, best_val_acc = float('inf'), 0.0

    for _ in range(epochs):
        model.train()
        for feats, labels in train_loader:
            feats, labels = feats.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(feats), labels)
            loss.backward(); optimizer.step()

        model.eval()
        val_loss = val_correct = val_total = 0
        with torch.no_grad():
            for feats, labels in val_loader:
                feats, labels = feats.to(device), labels.to(device)
                out  = model(feats)
                val_loss    += criterion(out, labels).item()
                val_correct += (torch.argmax(out, 1) == labels).sum().item()
                val_total   += labels.size(0)
        val_loss /= len(val_loader)
        val_acc   = val_correct / val_total
        if val_loss < best_val_loss:
            best_val_loss, best_val_acc = val_loss, val_acc

    return best_val_loss, best_val_acc


def create_fitness_function(X_train, y_train, X_val, y_val, num_classes):
    """Fitness = alpha*(1-acc) + beta*feature_ratio (minimise)."""
    def fitness(binary_mask):
        try:
            mask = binary_mask > 0.5
            if np.sum(mask) == 0:
                return 1.0
            X_tr_s = X_train[:, mask]
            X_v_s  = X_val[:,   mask]
            tr_ds  = MultiModalDataset(X_tr_s, y_train)
            v_ds   = MultiModalDataset(X_v_s,  y_val)
            tr_ld  = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
            v_ld   = DataLoader(v_ds,  batch_size=BATCH_SIZE)
            mdl    = SimpleNN(X_tr_s.shape[1], num_classes).to(DEVICE)
            _, val_acc = train_model_quick(mdl, tr_ld, v_ld, epochs=30, lr=1e-3, device=DEVICE)
            del mdl, tr_ds, v_ds, tr_ld, v_ld
            torch.cuda.empty_cache()
            alpha, beta = 0.95, 0.05
            return alpha * (1.0 - val_acc) + beta * (np.sum(mask) / len(mask))
        except Exception as e:
            print(f"Fitness error: {e}")
            return 1.0
    return fitness


def run_evolopy_optimizer(opt_name, opt_func, X_train, y_train, X_val, y_val,
                          num_classes, feature_dims):
    n_feats      = X_train.shape[1]
    fitness_func = create_fitness_function(X_train, y_train, X_val, y_val, num_classes)
    print(f"    Running {opt_name}  (pop={N_POPULATION}, iter={MAX_ITERATIONS}, feats={n_feats})")
    t0       = time.time()
    solution = opt_func(fitness_func, 0, 1, n_feats, N_POPULATION, MAX_ITERATIONS)
    mask     = solution.bestIndividual > 0.5
    ret      = calculate_modality_retention(mask, feature_dims)
    elapsed  = time.time() - t0
    print(f"      Selected {mask.sum()}/{n_feats} ({mask.sum()/n_feats*100:.1f}%)  "
          f"fitness={solution.convergence[-1]:.4f}  t={elapsed:.1f}s")
    return {
        'mask':               mask,
        'convergence':        solution.convergence,
        'best_fitness':       float(solution.convergence[-1]),
        'execution_time':     elapsed,
        'num_selected':       int(mask.sum()),
        'modality_retention': ret,
    }

print("Feature selection functions defined.")

Feature selection functions defined.


## Main Experiment: Train on UTD → Test on CZU

In [23]:
def run_cross_dataset_experiment():
    """
    1. Split UTD: all subjects except last -> train; last subject -> val.
    2. Normalise features (fit on UTD train only).
    3. For each method (baseline + 14 metaheuristics):
       a. Run feature selection on UTD train/val.
       b. Train final model on UTD train (selected features).
       c. Evaluate on UTD val AND CZU test (same scaler + mask applied).
    4. Save all results.
    """
    print("=" * 70)
    print("CROSS-DATASET EXPERIMENT: UTD (train) -> CZU (test)")
    print(f"Modalities: {SHARED_MODALITY_NAMES}  |  Methods: {len(META_METHODS)}")
    print("=" * 70)

    num_classes = len(np.unique(y_utd))

    # ── UTD train/val split ───────────────────────────────────────────────
    all_subjects = np.unique(subjects_utd)
    val_subject  = all_subjects[-1]
    val_idx      = np.where(subjects_utd == val_subject)[0]
    train_idx    = np.where(subjects_utd != val_subject)[0]
    print(f"\nUTD split: {len(train_idx)} train, {len(val_idx)} val  "
          f"(val subject: {val_subject})")

    # ── Preprocessing (fit on UTD train) ─────────────────────────────────
    print("\nPreprocessing UTD features...")
    X_utd_train, X_utd_val, feature_dims, scalers, pca_obj = prepare_data(
        X_utd, train_idx, val_idx, depth_pca_dim=DEPTH_PCA_DIM
    )
    total_features = sum(feature_dims.values())

    # Apply the same preprocessing to CZU (transform only — no re-fit)
    print("Applying UTD preprocessing to CZU test set...")
    X_czu_test = apply_preprocessing_to_czu(X_czu, scalers, pca_obj)
    print(f"CZU test set shape: {X_czu_test.shape}")

    results = {}

    # ── Loop over methods ────────────────────────────────────────────────
    for method_name in META_METHODS:
        print(f"\n{'─'*60}")
        print(f"  METHOD: {method_name.upper()}")
        print(f"{'─'*60}")

        # ── Feature selection ─────────────────────────────────────────
        fs_t0 = time.time()
        if method_name == 'baseline':
            feature_mask = np.ones(total_features, dtype=bool)
            fs_info = {'execution_time': 0, 'num_selected': total_features,
                       'convergence': None, 'best_fitness': None,
                       'modality_retention': {n: {'selected': feature_dims[n],
                                                  'total': feature_dims[n],
                                                  'percentage': 100.0}
                                              for n in SHARED_MODALITY_NAMES}}
        elif method_name.startswith('meta_'):
            opt_name = method_name.replace('meta_', '')
            opt_func = EVOLOPY_OPTIMIZERS[opt_name]
            fs_info  = run_evolopy_optimizer(
                opt_name, opt_func,
                X_utd_train, y_utd[train_idx],
                X_utd_val,   y_utd[val_idx],
                num_classes, feature_dims
            )
            feature_mask = fs_info['mask']
        else:
            raise ValueError(f"Unknown method: {method_name}")
        fs_time = time.time() - fs_t0

        # ── Apply mask ────────────────────────────────────────────────
        X_tr_sel  = X_utd_train[:, feature_mask]
        X_val_sel = X_utd_val[:,   feature_mask]
        X_czu_sel = X_czu_test[:,  feature_mask]

        # ── Train final model ─────────────────────────────────────────
        tr_ds  = MultiModalDataset(X_tr_sel,  y_utd[train_idx])
        val_ds = MultiModalDataset(X_val_sel, y_utd[val_idx])

        tr_ld  = DataLoader(tr_ds,  batch_size=BATCH_SIZE, shuffle=True)
        val_ld = DataLoader(val_ds, batch_size=BATCH_SIZE)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        model = SimpleNN(X_tr_sel.shape[1], UTD_NUM_CLASSES).to(DEVICE)
        model, train_info = train_model(model, tr_ld, val_ld, N_EPOCHS, LEARNING_RATE)

        # ── Evaluate ──────────────────────────────────────────────────
        # UTD val: standard eval (model trained on all 27 UTD classes)
        val_preds, val_true = evaluate_loader(model, val_ld)
        val_metrics = compute_metrics(val_preds, val_true)

        # CZU test: logit-masked eval — force argmax to clap/CW/CCW only,
        # then merge CW+CCW → draw_circle to get 2-class predictions
        czu_merged_preds, czu_raw_preds = evaluate_czu_with_logit_mask(
            model, X_czu_sel, UTD_NUM_CLASSES
        )
        czu_metrics = compute_metrics(czu_merged_preds, y_czu_remapped)

        outside = int(np.sum(czu_merged_preds == -1))
        print(f"    UTD val  acc={val_metrics['accuracy']*100:.2f}%  "
              f"f1={val_metrics['f1_macro']*100:.2f}%")
        print(f"    CZU test acc={czu_metrics['accuracy']*100:.2f}%  "
              f"f1={czu_metrics['f1_macro']*100:.2f}%  "
              f"feats={feature_mask.sum()}/{total_features} "
              f"({feature_mask.sum()/total_features*100:.1f}%)"
              + (f"  outside={outside}" if outside > 0 else ""))

        results[method_name] = {
            'method':               method_name,
            'feature_mask':         feature_mask,
            'num_features_selected': int(feature_mask.sum()),
            'num_features_total':   total_features,
            'feature_retention_pct': float(feature_mask.sum() / total_features * 100),
            'modality_retention':   fs_info['modality_retention'],
            'feature_dims':         feature_dims,
            'fs_execution_time':    fs_info['execution_time'],
            'fs_time_total':        fs_time,
            'train_time_sec':       train_info['train_time_sec'],
            'convergence':          fs_info.get('convergence'),
            'best_fitness':         fs_info.get('best_fitness'),
            'utd_val':              val_metrics,
            'czu_test':             czu_metrics,
            'model_params':         count_model_parameters(model),
            'model_size_mb':        get_model_size_mb(model),
            'gpu_mem_peak_mb':      train_info['gpu_mem_peak_mb'],
            'dataset_size_mb_orig': get_dataset_size_mb(X_utd_train),
            'dataset_size_mb_sel':  get_dataset_size_mb(X_tr_sel),
        }

        # ── Save immediately ──────────────────────────────────────────
        save = {k: v for k, v in results[method_name].items() if k != 'feature_mask'}
        def _clean(v):
            if isinstance(v, np.ndarray):   return v.tolist()
            if isinstance(v, (np.floating, np.integer)): return float(v)
            if isinstance(v, dict):
                return {kk: _clean(vv) for kk, vv in v.items()}
            return v
        clean = {k: _clean(v) for k, v in save.items()}
        with open(RESULTS_ROOT / method_name / "result.json", 'w') as f:
            json_lib.dump(clean, f, indent=2, default=str)
        np.save(RESULTS_ROOT / method_name / "mask.npy", feature_mask)

        del model, tr_ds, val_ds, tr_ld, val_ld
        torch.cuda.empty_cache(); gc.collect()

    print(f"\n{'='*70}")
    print("EXPERIMENT COMPLETE")
    print(f"{'='*70}")
    return results

print("Experiment runner defined.")

Experiment runner defined.


In [ ]:
results = run_cross_dataset_experiment()

CROSS-DATASET EXPERIMENT: UTD (train) -> CZU (test)
Modalities: ['depth', 'sensor', 'skeleton']  |  Methods: 15

UTD split: 754 train, 107 val  (val subject: 8)

Preprocessing UTD features...


    KernelPCA depth: 5508 -> 512
    Dims: depth=512 | sensor=652 | skeleton=1879 | TOTAL=3043
Applying UTD preprocessing to CZU test set...
CZU test set shape: (158, 3043)

────────────────────────────────────────────────────────────
  METHOD: BASELINE
────────────────────────────────────────────────────────────
    UTD val  acc=97.20%  f1=97.08%
    CZU test acc=52.53%  f1=44.84%  feats=3043/3043 (100.0%)

────────────────────────────────────────────────────────────
  METHOD: META_BAT
────────────────────────────────────────────────────────────
    Running BAT  (pop=20, iter=30, feats=3043)
BAT is optimizing  "fitness"
['At iteration 0 the best fitness is 0.04289667415026367']
['At iteration 1 the best fitness is 0.02545185672034177']
['At iteration 2 the best fitness is 0.02545185672034177']
['At iteration 3 the best fitness is 0.02545185672034177']
['At iteration 4 the best fitness is 0.02545185672034177']
['At iteration 5 the best fitness is 0.02545185672034177']
['At iteration 6 

## Run from MVO

In [11]:
EVOLOPY_OPTIMIZERS_from_MVO = {
    'MVO':  MVO.MVO,
    'PSO':  PSO.PSO,
    'SCA':  SCA.SCA,
    'SSA':  SSA.SSA,
    'WOA':  WOA.WOA,
}

META_METHODS_from_MVO = ['baseline'] + [f'meta_{k}' for k in EVOLOPY_OPTIMIZERS_from_MVO.keys()]

In [12]:
def run_cross_dataset_experiment_from_MVO():
    """
    1. Split UTD: all subjects except last -> train; last subject -> val.
    2. Normalise features (fit on UTD train only).
    3. For each method (baseline + 14 metaheuristics):
       a. Run feature selection on UTD train/val.
       b. Train final model on UTD train (selected features).
       c. Evaluate on UTD val AND CZU test (same scaler + mask applied).
    4. Save all results.
    """
    print("=" * 70)
    print("CROSS-DATASET EXPERIMENT: UTD (train) -> CZU (test)")
    print(f"Modalities: {SHARED_MODALITY_NAMES}  |  Methods: {len(META_METHODS_from_MVO)}")
    print("=" * 70)

    num_classes = len(np.unique(y_utd))

    # ── UTD train/val split ───────────────────────────────────────────────
    all_subjects = np.unique(subjects_utd)
    val_subject  = all_subjects[-1]
    val_idx      = np.where(subjects_utd == val_subject)[0]
    train_idx    = np.where(subjects_utd != val_subject)[0]
    print(f"\nUTD split: {len(train_idx)} train, {len(val_idx)} val  "
          f"(val subject: {val_subject})")

    # ── Preprocessing (fit on UTD train) ─────────────────────────────────
    print("\nPreprocessing UTD features...")
    X_utd_train, X_utd_val, feature_dims, scalers, pca_obj = prepare_data(
        X_utd, train_idx, val_idx, depth_pca_dim=DEPTH_PCA_DIM
    )
    total_features = sum(feature_dims.values())

    # Apply the same preprocessing to CZU (transform only — no re-fit)
    print("Applying UTD preprocessing to CZU test set...")
    X_czu_test = apply_preprocessing_to_czu(X_czu, scalers, pca_obj)
    print(f"CZU test set shape: {X_czu_test.shape}")

    results = {}

    # ── Loop over methods ────────────────────────────────────────────────
    for method_name in META_METHODS_from_MVO:
        print(f"\n{'─'*60}")
        print(f"  METHOD: {method_name.upper()}")
        print(f"{'─'*60}")

        # ── Feature selection ─────────────────────────────────────────
        fs_t0 = time.time()
        if method_name == 'baseline':
            feature_mask = np.ones(total_features, dtype=bool)
            fs_info = {'execution_time': 0, 'num_selected': total_features,
                       'convergence': None, 'best_fitness': None,
                       'modality_retention': {n: {'selected': feature_dims[n],
                                                  'total': feature_dims[n],
                                                  'percentage': 100.0}
                                              for n in SHARED_MODALITY_NAMES}}
        elif method_name.startswith('meta_'):
            opt_name = method_name.replace('meta_', '')
            opt_func = EVOLOPY_OPTIMIZERS_from_MVO[opt_name]
            fs_info  = run_evolopy_optimizer(
                opt_name, opt_func,
                X_utd_train, y_utd[train_idx],
                X_utd_val,   y_utd[val_idx],
                num_classes, feature_dims
            )
            feature_mask = fs_info['mask']
        else:
            raise ValueError(f"Unknown method: {method_name}")
        fs_time = time.time() - fs_t0

        # ── Apply mask ────────────────────────────────────────────────
        X_tr_sel  = X_utd_train[:, feature_mask]
        X_val_sel = X_utd_val[:,   feature_mask]
        X_czu_sel = X_czu_test[:,  feature_mask]

        # ── Train final model ─────────────────────────────────────────
        tr_ds  = MultiModalDataset(X_tr_sel,  y_utd[train_idx])
        val_ds = MultiModalDataset(X_val_sel, y_utd[val_idx])

        tr_ld  = DataLoader(tr_ds,  batch_size=BATCH_SIZE, shuffle=True)
        val_ld = DataLoader(val_ds, batch_size=BATCH_SIZE)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        model = SimpleNN(X_tr_sel.shape[1], UTD_NUM_CLASSES).to(DEVICE)
        model, train_info = train_model(model, tr_ld, val_ld, N_EPOCHS, LEARNING_RATE)

        # ── Evaluate ──────────────────────────────────────────────────
        # UTD val: standard eval (model trained on all 27 UTD classes)
        val_preds, val_true = evaluate_loader(model, val_ld)
        val_metrics = compute_metrics(val_preds, val_true)

        # CZU test: logit-masked eval — force argmax to clap/CW/CCW only,
        # then merge CW+CCW → draw_circle to get 2-class predictions
        czu_merged_preds, czu_raw_preds = evaluate_czu_with_logit_mask(
            model, X_czu_sel, UTD_NUM_CLASSES
        )
        czu_metrics = compute_metrics(czu_merged_preds, y_czu_remapped)

        outside = int(np.sum(czu_merged_preds == -1))
        print(f"    UTD val  acc={val_metrics['accuracy']*100:.2f}%  "
              f"f1={val_metrics['f1_macro']*100:.2f}%")
        print(f"    CZU test acc={czu_metrics['accuracy']*100:.2f}%  "
              f"f1={czu_metrics['f1_macro']*100:.2f}%  "
              f"feats={feature_mask.sum()}/{total_features} "
              f"({feature_mask.sum()/total_features*100:.1f}%)"
              + (f"  outside={outside}" if outside > 0 else ""))

        results[method_name] = {
            'method':               method_name,
            'feature_mask':         feature_mask,
            'num_features_selected': int(feature_mask.sum()),
            'num_features_total':   total_features,
            'feature_retention_pct': float(feature_mask.sum() / total_features * 100),
            'modality_retention':   fs_info['modality_retention'],
            'feature_dims':         feature_dims,
            'fs_execution_time':    fs_info['execution_time'],
            'fs_time_total':        fs_time,
            'train_time_sec':       train_info['train_time_sec'],
            'convergence':          fs_info.get('convergence'),
            'best_fitness':         fs_info.get('best_fitness'),
            'utd_val':              val_metrics,
            'czu_test':             czu_metrics,
            'model_params':         count_model_parameters(model),
            'model_size_mb':        get_model_size_mb(model),
            'gpu_mem_peak_mb':      train_info['gpu_mem_peak_mb'],
            'dataset_size_mb_orig': get_dataset_size_mb(X_utd_train),
            'dataset_size_mb_sel':  get_dataset_size_mb(X_tr_sel),
        }

        # ── Save immediately ──────────────────────────────────────────
        save = {k: v for k, v in results[method_name].items() if k != 'feature_mask'}
        def _clean(v):
            if isinstance(v, np.ndarray):   return v.tolist()
            if isinstance(v, (np.floating, np.integer)): return float(v)
            if isinstance(v, dict):
                return {kk: _clean(vv) for kk, vv in v.items()}
            return v
        clean = {k: _clean(v) for k, v in save.items()}
        with open(RESULTS_ROOT / method_name / "result.json", 'w') as f:
            json_lib.dump(clean, f, indent=2, default=str)
        np.save(RESULTS_ROOT / method_name / "mask.npy", feature_mask)

        del model, tr_ds, val_ds, tr_ld, val_ld
        torch.cuda.empty_cache(); gc.collect()

    print(f"\n{'='*70}")
    print("EXPERIMENT COMPLETE")
    print(f"{'='*70}")
    return results

print("Experiment runner defined.")

Experiment runner defined.


In [13]:
results_from_MVO = run_cross_dataset_experiment_from_MVO()

CROSS-DATASET EXPERIMENT: UTD (train) -> CZU (test)
Modalities: ['depth', 'sensor', 'skeleton']  |  Methods: 6

UTD split: 754 train, 107 val  (val subject: 8)

Preprocessing UTD features...
    KernelPCA depth: 5508 -> 512
    Dims: depth=512 | sensor=652 | skeleton=1879 | TOTAL=3043
Applying UTD preprocessing to CZU test set...
CZU test set shape: (158, 3043)

────────────────────────────────────────────────────────────
  METHOD: BASELINE
────────────────────────────────────────────────────────────
    UTD val  acc=97.20%  f1=97.08%
    CZU test acc=52.53%  f1=44.84%  feats=3043/3043 (100.0%)

────────────────────────────────────────────────────────────
  METHOD: META_MVO
────────────────────────────────────────────────────────────
    Running MVO  (pop=20, iter=30, feats=3043)
MVO is optimizing  "fitness"
['At iteration 1 the best fitness is 0.04289667415026367']
['At iteration 2 the best fitness is 0.04235444608585355']
['At iteration 3 the best fitness is 0.033558097180291215']
['

## Reconstruct full 'results' variable

In [14]:
# ── Reconstruct full `results` dict from saved per-method files ──────────────
import json as json_lib
import numpy as np
from pathlib import Path

results = {}

for method_name in META_METHODS:
    method_dir = RESULTS_ROOT / method_name
    json_path  = method_dir / "result.json"
    mask_path  = method_dir / "mask.npy"

    if not json_path.exists() or not mask_path.exists():
        print(f"  [SKIP] {method_name}: missing file(s) in {method_dir}")
        continue

    with open(json_path) as f:
        r = json_lib.load(f)

    # Restore numpy arrays from lists (convergence is stored as list)
    if r.get('convergence') is not None:
        r['convergence'] = list(r['convergence'])   # already a list — keep as-is

    # Re-attach the binary mask as a numpy bool array
    r['feature_mask'] = np.load(mask_path)

    results[method_name] = r
    print(f"  [OK] {method_name:20s}  "
          f"feats={r['num_features_selected']}/{r['num_features_total']}  "
          f"czu_acc={r['czu_test']['accuracy']*100:.2f}%")

# Merge MVO-and-later results from the fresh run
for method_name, r in results_from_MVO.items():
    results[method_name] = r
    print(f"  [MERGED from live run] {method_name}")

print(f"\nTotal methods loaded: {len(results)} / {len(META_METHODS)}")
missing = [m for m in META_METHODS if m not in results]
if missing:
    print(f"Still missing: {missing}")

  [OK] baseline              feats=3043/3043  czu_acc=52.53%
  [OK] meta_BAT              feats=1547/3043  czu_acc=56.33%
  [OK] meta_CS               feats=1475/3043  czu_acc=62.03%
  [OK] meta_DE               feats=1437/3043  czu_acc=63.29%
  [OK] meta_FFA              feats=1499/3043  czu_acc=57.59%
  [OK] meta_GA               feats=1480/3043  czu_acc=61.39%
  [OK] meta_GWO              feats=1244/3043  czu_acc=53.80%
  [OK] meta_HHO              feats=322/3043  czu_acc=52.53%
  [OK] meta_JAYA             feats=1471/3043  czu_acc=55.06%
  [OK] meta_MFO              feats=1483/3043  czu_acc=66.46%
  [OK] meta_MVO              feats=1496/3043  czu_acc=52.53%
  [OK] meta_PSO              feats=1488/3043  czu_acc=43.04%
  [OK] meta_SCA              feats=1505/3043  czu_acc=53.16%
  [OK] meta_SSA              feats=1472/3043  czu_acc=32.28%
  [OK] meta_WOA              feats=745/3043  czu_acc=35.44%
  [MERGED from live run] baseline
  [MERGED from live run] meta_MVO
  [MERGED from live

## Results Table

In [15]:
rows = []
for method_name, r in results.items():
    row = {
        'Method':                   method_name,
        # CZU test (primary metric — cross-dataset generalisation)
        'CZU Test Acc (%)':         r['czu_test']['accuracy']        * 100,
        'CZU Test F1 Macro (%)':    r['czu_test']['f1_macro']        * 100,
        'CZU Test F1 Weighted (%)': r['czu_test']['f1_weighted']     * 100,
        'CZU Precision Macro (%)':  r['czu_test']['precision_macro'] * 100,
        'CZU Recall Macro (%)':     r['czu_test']['recall_macro']    * 100,
        # UTD val (sanity check)
        'UTD Val Acc (%)':          r['utd_val']['accuracy']         * 100,
        'UTD Val F1 Macro (%)':     r['utd_val']['f1_macro']         * 100,
        # Feature selection
        'Features Selected':        r['num_features_selected'],
        'Features Total':           r['num_features_total'],
        'Feature Retention (%)':    r['feature_retention_pct'],
        # Timing
        'FS Time (s)':              r['fs_execution_time'],
        'Train Time (s)':           r['train_time_sec'],
        # Model
        'Model Params':             r['model_params'],
        'Model Size (MB)':          r['model_size_mb'],
        'GPU Peak (MB)':            r['gpu_mem_peak_mb'],
    }
    # Per-modality retention
    for mod in SHARED_MODALITY_NAMES:
        if mod in r.get('modality_retention', {}):
            row[f'{mod}_retention_%'] = r['modality_retention'][mod]['percentage']
    rows.append(row)

df = pd.DataFrame(rows).round(2)
df.to_csv(RESULTS_ROOT / "cross_dataset_results.csv", index=False)
print(df[['Method', 'CZU Test Acc (%)', 'CZU Test F1 Macro (%)',
          'UTD Val Acc (%)', 'Feature Retention (%)']].to_string(index=False))

   Method  CZU Test Acc (%)  CZU Test F1 Macro (%)  UTD Val Acc (%)  Feature Retention (%)
 baseline             52.53                  44.84            97.20                 100.00
 meta_BAT             56.33                  47.43            99.07                  50.84
  meta_CS             62.03                  57.80            99.07                  48.47
  meta_DE             63.29                  53.67            98.13                  47.22
 meta_FFA             57.59                  39.14           100.00                  49.26
  meta_GA             61.39                  43.45           100.00                  48.64
 meta_GWO             53.80                  45.06           100.00                  40.88
 meta_HHO             52.53                  47.01            96.26                  10.58
meta_JAYA             55.06                  53.71            98.13                  48.34
 meta_MFO             66.46                  57.35            99.07                  48.73

## Visualisations

In [16]:
# ── Plot 1: CZU Test Accuracy — Baseline vs Metaheuristics ─────────────────
fig, ax = plt.subplots(figsize=(16, 6))

df_sorted = df.sort_values('CZU Test Acc (%)', ascending=False)
colors = ['#2196F3' if m == 'baseline' else '#FF5722' for m in df_sorted['Method']]
bars = ax.bar(df_sorted['Method'], df_sorted['CZU Test Acc (%)'],
              color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, df_sorted['CZU Test Acc (%)']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.axhline(df[df['Method'] == 'baseline']['CZU Test Acc (%)'].values[0],
           color='#2196F3', linestyle='--', linewidth=1.5, alpha=0.7, label='Baseline')
ax.set_ylabel('CZU Test Accuracy (%)', fontsize=12)
ax.set_title('Cross-Dataset Accuracy: Trained on UTD-MHAD, Tested on CZU-MHAD\n'
             '(Baseline vs. Metaheuristic Feature Selection)', fontsize=13)
ax.legend(fontsize=10)
plt.xticks(rotation=45, ha='right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '01_czu_test_accuracy.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 01_czu_test_accuracy.png")

Saved: 01_czu_test_accuracy.png


In [17]:
# ── Plot 2: CZU F1 Macro vs Feature Retention scatter ───────────────────────
fig, ax = plt.subplots(figsize=(12, 7))

baseline_row = df[df['Method'] == 'baseline'].iloc[0]
meta_df = df[df['Method'] != 'baseline']

ax.scatter(meta_df['Feature Retention (%)'], meta_df['CZU Test F1 Macro (%)'],
           s=100, c='#FF5722', alpha=0.8, zorder=5, label='Metaheuristics')
ax.scatter(baseline_row['Feature Retention (%)'], baseline_row['CZU Test F1 Macro (%)'],
           s=200, c='#2196F3', marker='*', zorder=6, label='Baseline')

for _, row in meta_df.iterrows():
    ax.annotate(row['Method'].replace('meta_', ''),
                (row['Feature Retention (%)'], row['CZU Test F1 Macro (%)']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)

ax.axhline(baseline_row['CZU Test F1 Macro (%)'], color='#2196F3',
           linestyle='--', alpha=0.5, label='Baseline F1')
ax.set_xlabel('Feature Retention (%)', fontsize=12)
ax.set_ylabel('CZU Test F1 Macro (%)', fontsize=12)
ax.set_title('Accuracy–Retention Trade-off on CZU-MHAD Test Set', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '02_f1_vs_retention_scatter.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 02_f1_vs_retention_scatter.png")

Saved: 02_f1_vs_retention_scatter.png


In [18]:
# ── Plot 3: UTD Val vs CZU Test Accuracy side-by-side ───────────────────────
fig, ax = plt.subplots(figsize=(16, 7))

x   = np.arange(len(df))
w   = 0.38
ax.bar(x - w/2, df['UTD Val Acc (%)'],  w, label='UTD Val Acc',  color='#42A5F5', alpha=0.85)
ax.bar(x + w/2, df['CZU Test Acc (%)'], w, label='CZU Test Acc', color='#FF7043', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([m.replace('meta_', '') for m in df['Method']],
                   rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('UTD Val Accuracy vs CZU Test Accuracy per Method\n'
             '(Cross-dataset generalisation gap)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '03_utd_val_vs_czu_test.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 03_utd_val_vs_czu_test.png")

Saved: 03_utd_val_vs_czu_test.png


In [19]:
# ── Plot 4: Feature Retention per Metaheuristic ─────────────────────────────
meta_df = df[df['Method'] != 'baseline'].copy()
meta_df = meta_df.sort_values('Feature Retention (%)')

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.RdYlGn(meta_df['CZU Test Acc (%)'] / meta_df['CZU Test Acc (%)'].max())
bars = ax.barh(meta_df['Method'].str.replace('meta_', ''),
               meta_df['Feature Retention (%)'],
               color=colors, alpha=0.85, edgecolor='white')

for bar, val in zip(bars, meta_df['Feature Retention (%)']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

ax.axvline(100, color='#2196F3', linestyle='--', alpha=0.6, label='Baseline (100%)')
ax.set_xlabel('Feature Retention (%)', fontsize=12)
ax.set_title('Feature Retention per Metaheuristic\n'
             '(colour = CZU test accuracy, green=high)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '04_feature_retention.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 04_feature_retention.png")

Saved: 04_feature_retention.png


In [21]:
# ── Plot 6: Per-modality Retention Heatmap ───────────────────────────────────
mod_cols = [f'{n}_retention_%' for n in SHARED_MODALITY_NAMES]
hm_df = df.set_index('Method')[mod_cols].rename(
    columns={f'{n}_retention_%': n for n in SHARED_MODALITY_NAMES})

fig, ax = plt.subplots(figsize=(7, 10))
sns.heatmap(hm_df, annot=True, fmt='.1f', cmap='RdYlGn',
            vmin=0, vmax=100, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Retention (%)'})
ax.set_title('Per-modality Feature Retention (%)', fontsize=13)
ax.set_xlabel('Modality'); ax.set_ylabel('Method')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '06_modality_retention_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 06_modality_retention_heatmap.png")

Saved: 06_modality_retention_heatmap.png


## Summary Statistics

In [24]:
baseline_czu = df[df['Method'] == 'baseline']['CZU Test Acc (%)'].values[0]
baseline_f1  = df[df['Method'] == 'baseline']['CZU Test F1 Macro (%)'].values[0]

meta_df = df[df['Method'] != 'baseline'].copy()
best_row = meta_df.loc[meta_df['CZU Test Acc (%)'].idxmax()]
worst_row = meta_df.loc[meta_df['CZU Test Acc (%)'].idxmin()]

print("=" * 60)
print("CROSS-DATASET RESULTS SUMMARY")
print("=" * 60)
print(f"\nBaseline (no FS):")
print(f"  CZU Test Accuracy : {baseline_czu:.2f}%")
print(f"  CZU Test F1 Macro : {baseline_f1:.2f}%")

print(f"\nBest Metaheuristic: {best_row['Method']}")
print(f"  CZU Test Accuracy : {best_row['CZU Test Acc (%)']:.2f}%  "
      f"({'+'if best_row['CZU Test Acc (%)'] > baseline_czu else ''}"
      f"{best_row['CZU Test Acc (%)'] - baseline_czu:.2f}% vs baseline)")
print(f"  CZU Test F1 Macro : {best_row['CZU Test F1 Macro (%)']:.2f}%")
print(f"  Feature Retention : {best_row['Feature Retention (%)']:.1f}%")

CROSS-DATASET RESULTS SUMMARY

Baseline (no FS):
  CZU Test Accuracy : 52.53%
  CZU Test F1 Macro : 44.84%

Best Metaheuristic: meta_MFO
  CZU Test Accuracy : 66.46%  (+13.93% vs baseline)
  CZU Test F1 Macro : 57.35%
  Feature Retention : 48.7%
